In [1]:
import os

os.getcwd()

'c:\\Users\\divya\\Desktop\\proj\\ml-ops\\dsproject\\research'

In [2]:
os.chdir("../")

In [3]:
os.getcwd()

'c:\\Users\\divya\\Desktop\\proj\\ml-ops\\dsproject'

In [4]:
from dataclasses import dataclass
from pathlib import Path
from src.dsproject.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from src.dsproject.utils.common import read_yaml, create_directories
import urllib.request as requests
import zipfile
import os
from src.dsproject import logger

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

class ConfiguartionManager:
    def __init__(self, config_file_path: Path = CONFIG_FILE_PATH, 
                 params_file_path: Path = PARAMS_FILE_PATH,
                 schema_file_path: Path = SCHEMA_FILE_PATH):
        
        self.config_info = read_yaml(config_file_path)
        self.params_info = read_yaml(params_file_path)
        self.schema_info = read_yaml(schema_file_path)

        create_directories([self.config_info.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config_info.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )
        
        return data_ingestion_config
    
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_data(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = requests.urlretrieve(url = self.config.source_URL, filename = self.config.local_data_file)
            logger.info(f"Data downloaded successfully from {self.config.source_URL} to {self.config.local_data_file}")

        else:
            logger.info(f"Data already exists at {self.config.local_data_file}. Skipping download.")

    def extract_zip_file(self):
        unzip_dir = self.config.unzip_dir
        if not os.path.exists(unzip_dir):
            os.makedirs(unzip_dir)
            
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(self.config.unzip_dir)
            logger.info(f"Data extracted successfully to {self.config.unzip_dir}")


In [5]:
try:
    config_manager = ConfiguartionManager()
    data_ingestion_config = config_manager.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_data()
    data_ingestion.extract_zip_file()
except Exception as e:
    logger.exception(f"An error occurred during data ingestion: {e}")

[2026-04-23 16:51:57,664: INFO: common: Reading YAML file from: config\config.yaml]
[2026-04-23 16:51:57,697: INFO: common: Successfully read YAML file: config\config.yaml]
[2026-04-23 16:51:57,701: INFO: common: Reading YAML file from: params.yaml]
[2026-04-23 16:51:57,704: INFO: common: Successfully read YAML file: params.yaml]
[2026-04-23 16:51:57,707: INFO: common: Reading YAML file from: schema.yaml]
[2026-04-23 16:51:57,709: INFO: common: Successfully read YAML file: schema.yaml]
[2026-04-23 16:51:57,714: INFO: common: Creating directory at: artifacts]
[2026-04-23 16:51:57,718: INFO: common: Successfully created directory: artifacts]
[2026-04-23 16:51:57,720: INFO: common: Creating directory at: artifacts/data_ingestion]
[2026-04-23 16:51:57,724: INFO: common: Successfully created directory: artifacts/data_ingestion]
[2026-04-23 16:51:59,408: INFO: 2671723942: Data downloaded successfully from https://github.com/krishnaik06/datasets/raw/main/winequality-data.zip to artifacts\data